[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance2_cours.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- transformer des colonnes de texte en variables utilisables par un modèle
- ajuster une régression logistique et lire une probabilité de départ
- expliquer pourquoi la justesse est un piège sur des données déséquilibrées
- lire une matrice de confusion, la précision et le rappel
- choisir un seuil de décision à partir d'un coût, pas d'une habitude

## Un opérateur, 7 043 abonnés, une question à 30 millions

Nouveau terrain : un opérateur télécom. Chaque ligne est un abonné, et la
colonne `churn` vaut **1 s'il a résilié**, 0 sinon.

> *« Qui va partir le mois prochain — et qui faut-il appeler ? »*

C'est une **prédiction binaire** : la réponse n'est plus un nombre, c'est une
décision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")   ## une ligne = un abonne

print(tel.shape)
tel.head(3)

### D'abord, nettoyer — rien n'a changé depuis le bloc 2

`total` est arrivé en texte : onze abonnés tout neufs n'ont pas encore de
facture cumulée.

In [ ]:
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")   ## coerce -> NaN
print(tel["total"].isna().sum(), "valeurs vides ->", end=" ")   ## on VERIFIE

tel = tel.dropna(subset=["total"])   ## cette colonne seulement
print(len(tel), "abonnes conserves")

In [ ]:
# La moyenne d'une colonne de 0 et de 1, c'est la proportion de 1
print("taux de resiliation :", round(100 * tel["churn"].mean(), 1), "%")

(tel.groupby("contrat")["churn"].mean() * 100).round(1)   ## par type de contrat

**42,7 % chez les abonnés au mois, 2,8 % chez ceux engagés deux ans.**

Gardez ce tableau en tête : la séance 4.3 y reviendra pour en faire une
recommandation chiffrée.

## 1. Du texte vers des nombres

Un modèle multiplie des nombres : il ne sait pas quoi faire de `"mensuel"`.
`get_dummies` transforme chaque modalité en une colonne 0/1.

In [ ]:
y = tel["churn"]   ## la cible : 1 = parti
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True)

print(X.shape[1], "colonnes apres transformation")   ## 9 -> 14
list(X.columns)

> 💡 `drop_first=True` supprime une modalité par variable, celle qui devient
> la **référence** — exactement comme la modalité absente du tableau de
> régression de la séance 3.4. Si ce n'est ni `un_an` ni `deux_ans`, c'est
> forcément `mensuel` : la troisième colonne n'apporterait rien.

## 2. Ajuster une régression logistique

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)   ## stratify : voir plus bas

# stratify=y : garder le meme taux de resiliation des deux cotes
print(round(100 * y_tr.mean(), 1), "% de churn en apprentissage |",
      round(100 * y_te.mean(), 1), "% en test")

In [ ]:
# StandardScaler d'abord : l'anciennete va de 0 a 72, la facture cumulee
# de 18 a 8 700. Sans mise a l'echelle, l'optimisation converge mal.
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modele.fit(X_tr, y_tr)

# predict_proba renvoie deux colonnes : [proba de rester, proba de partir]
proba = modele.predict_proba(X_te)[:, 1]   ## l'indice 1 : la proba de partir
print("probabilite de depart des 5 premiers :", proba[:5].round(3))

La logistique ne répond pas « part » ou « reste » : elle donne une
**probabilité de départ**. C'est nous qui déciderons à partir de quel niveau
on agit — et ce choix est le sujet de la fin de séance.

## 3. La justesse est un piège

In [ ]:
pred = modele.predict(X_te)   ## predict tranche a 0,50 par defaut

print("justesse du modele :", round(100 * accuracy_score(y_te, pred), 1), "%")

79,8 %. Quatre bonnes réponses sur cinq — de quoi présenter le projet en
comité.

**Avant ça, une question :** quelle justesse obtiendrait un modèle qui prédit
que *personne* ne part ?

In [ ]:
# Le modele le plus bete du monde : il repond toujours "reste"
print("toujours predire 'reste' :", round(100 * (1 - y_te.mean()), 1), "%")   ## 73,4

**73,4 %.** Notre modèle ne gagne que **six points** sur un modèle qui ne fait
strictement rien — et qui, lui, ne sauverait aucun client.

C'est le piège des données **déséquilibrées** : quand une classe pèse trois
quarts du fichier, la justesse récompense le fait de toujours parier dessus.

### Ce que la justesse cachait

In [ ]:
pd.DataFrame(confusion_matrix(y_te, pred),
             index=["reste vraiment", "part vraiment"],   ## la verite, en lignes
             columns=["predit reste", "predit part"])     ## le modele, en colonnes

Lisez la case en bas à gauche : **255 abonnés sont partis sans qu'on ait rien
tenté**. C'est le chiffre qui coûte de l'argent, et la justesse ne le montrait
pas.

Deux mesures le disent, elles :

In [ ]:
print("precision :", round(precision_score(y_te, pred), 3),
      "- part de vrais partants parmi ceux qu'on contacte")
print("rappel    :", round(recall_score(y_te, pred), 3),
      "- part des partants qu'on a retrouves")
print("AUC       :", round(roc_auc_score(y_te, proba), 3))

- **Rappel 0,545** : nous retrouvons un partant sur deux. L'autre moitié s'en
  va sans que personne ne l'appelle.
- **Précision 0,640** : sur dix abonnés contactés, six allaient vraiment
  partir, quatre non — quatre appels pour rien.
- **AUC 0,834** : la qualité du modèle **à tous les seuils à la fois**. Elle
  ne dépend pas du seuil choisi, contrairement aux deux autres.

## 4. Le seuil est une décision de gestion

`predict` a tranché à **0,50**, parce que c'est la valeur par défaut. Rien ne
l'impose. Regardons ce que d'autres seuils donnent.

In [ ]:
for seuil in [0.5, 0.4, 0.3, 0.2]:
    p = (proba > seuil).astype(int)   ## True/False -> 1/0
    print(f"seuil {seuil} : rappel {recall_score(y_te, p):.2f}  "
          f"precision {precision_score(y_te, p):.2f}  "
          f"contacts {p.sum()}")

Baisser le seuil retrouve plus de partants (rappel ↑) au prix de plus
d'appels inutiles (précision ↓). **Aucun des deux réglages n'est « le bon »
en soi** — cela dépend de ce que coûte chaque erreur.

### Chiffrons

- un appel de rétention coûte **15 €**
- un client retenu rapporte **300 €** de marge sur l'année
- une relance convainc environ **30 %** des partants contactés

In [ ]:
for seuil in [0.5, 0.4, 0.3, 0.2, 0.1]:
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()   ## partants effectivement rattrapes
    gain = vrais * 0.30 * 300 - p.sum() * 15   ## 30 % convaincus, 15 EUR l'appel
    print(f"seuil {seuil} : {p.sum():>4} appels, {vrais:>3} vrais partants "
          f"-> {gain:>8.0f} euros")

**20 370 € au seuil par défaut, 28 365 € au seuil 0,20.**

Huit mille euros de plus, sans toucher une ligne du modèle. Le réglage qui
rapportait le plus n'était pas dans l'algorithme, il était dans la décision.

> ⚠️ **Ne laissez jamais `predict` choisir à votre place.** Son seuil de 0,50
> est une convention informatique, pas un arbitrage économique. Dès qu'une
> erreur coûte plus cher que l'autre, le bon seuil se calcule.

## 5. Deux erreurs, dont une qui ne prévient pas

### L'erreur bruyante

In [ ]:
LogisticRegression().fit(tel.drop(columns=["churn"]), y)   ## sans dummies

Dernière ligne :

```
ValueError: could not convert string to float: 'mensuel'
```

On a donné les colonnes de texte telles quelles. Il manque le `get_dummies`.

### L'erreur silencieuse

In [ ]:
nul = pd.Series(0, index=y_te.index)   ## "personne ne part", jamais

print("justesse :", round(100 * accuracy_score(y_te, nul), 1), "%")
print("rappel   :", round(recall_score(y_te, nul), 3))

**73,4 % de justesse, et un rappel de 0.**

Un modèle qui ne prédit jamais aucun départ affiche un chiffre parfaitement
présentable. Rien dans ce 73,4 % ne dit qu'il est inutile.

> ⚠️ **Devant tout modèle de classification, exigez trois choses :** le score
> du modèle nul, la matrice de confusion, et le rappel. Une justesse seule ne
> veut rien dire.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| du texte en colonnes numériques | `pd.get_dummies(X, drop_first=True)` |
| enchaîner mise à l'échelle et modèle | `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` |
| une décision (0 ou 1) | `m.predict(X_te)` |
| une **probabilité** | `m.predict_proba(X_te)[:, 1]` |
| la matrice de confusion | `confusion_matrix(y_te, pred)` |
| la part de vrais parmi les prédits partants | `precision_score(y_te, pred)` |
| la part de partants retrouvés | `recall_score(y_te, pred)` |
| la qualité à tous les seuils | `roc_auc_score(y_te, proba)` |

## La matrice de confusion, en clair

|  | prédit : reste | prédit : part |
|---|---|---|
| **reste vraiment** | bien vu | fausse alerte — un contact pour rien |
| **part vraiment** | **client perdu sans rien tenter** | bien vu |

## Les trois phrases à retenir

1. **La justesse est un piège.** Prédire « personne ne part » donne 73,4 % de
   bonnes réponses et zéro client sauvé.

2. **Précision et rappel arbitrent deux coûts différents.** Le rappel dit
   combien de partants on retrouve, la précision combien de contacts sont
   utiles. On ne maximise pas les deux.

3. **Le seuil est une décision de gestion.** Le déplacer de 0,50 à 0,20 fait
   passer le gain de la campagne de 20 370 € à 28 365 € — sans changer une
   ligne du modèle.